# CIFAR-10 CNN 분류기

In [7]:
import torch

In [8]:
from typing import Tuple

from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

## 1. 데이터셋

In [9]:
# CIFAR-10: 32x32 컬러(3채널), 10개 클래스
# 학습셋에만 증강을 주고, 정규화는 양쪽 동일하게
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

cifar_train = datasets.CIFAR10(root="./cifar_data", train=True, download=True, transform=train_transform)
cifar_test = datasets.CIFAR10(root="./cifar_data", train=False, download=True, transform=test_transform)

# 모델 생성 셀에서 len(label_names) 로 쓰이는 값
label_names = cifar_train.classes
print(label_names)

c:\Users\ljs44\anaconda3\envs\py311\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [10]:
print(len(cifar_train), len(cifar_test))
print(cifar_train[0][0].shape, cifar_train[0][1])  # torch.Size([3, 32, 32]) 6

50000 10000
torch.Size([3, 32, 32]) 6


## 2. DataLoader

In [11]:
batch_size = 128
train_loader = DataLoader(cifar_train, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(cifar_test, batch_size=batch_size, shuffle=False)

images, labels = next(iter(train_loader))
print(images.shape, labels.shape)  # torch.Size([128, 3, 32, 32]) torch.Size([128])

torch.Size([128, 3, 32, 32]) torch.Size([128])


## 3. 모델

conv/pool 를 거치며 feature map 이 이렇게 줄어든다:

`(3,32,32)` -> conv_1 -> pool_1 `(32,16,16)` -> conv_2 -> pool_2 `(64,8,8)`
-> conv_3 (padding 없음) `(128,6,6)` -> pool_3 `(128,3,3)` -> conv_4 (padding 없음) `(128,1,1)`
-> flatten `(128,)` -> fc_1 -> fc_2 `(10,)`

In [12]:
class CNNClassifier(nn.Module):
    def __init__(self, output_dim: int):
        super(CNNClassifier, self).__init__()

        self.conv_1 = nn.Conv2d(3, 32, kernel_size=5, stride=1, padding=2)

        self.pool_1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv_2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)

        self.pool_2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv_3 = nn.Conv2d(64, 128, kernel_size=3, stride=1)

        self.pool_3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv_4 = nn.Conv2d(128, 128, kernel_size=3, stride=1)

        self.fc_1 = nn.Linear(128, 128)
        self.fc_2 = nn.Linear(128, output_dim, bias=False)

        self.dropout = nn.Dropout(0.3)
        self.dropout_2d = nn.Dropout2d(0.1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv_1(x)
        x = self.dropout_2d(F.relu(self.pool_1(x)))

        x = self.conv_2(x)
        x = self.dropout_2d(F.relu(self.pool_2(x)))

        x = self.conv_3(x)
        x = self.dropout_2d(F.relu(self.pool_3(x)))

        x = self.conv_4(x)
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(x))

        x = self.fc_1(x)
        x = self.dropout(F.relu(x))

        x = self.fc_2(x)
        return x

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_epoch = 20
learning_rate = 1e-3
lr_decay = 0.9

model = CNNClassifier(len(label_names)).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, lr_decay)
criterion = nn.CrossEntropyLoss()

In [14]:
def evaluate(model: nn.Module, dataloader: DataLoader, criterion: nn.Module, device: torch.device) -> Tuple[float, float]:
    avg_loss = torch.tensor(0.0, device=device)
    avg_accuracy = torch.tensor(0.0, device=device)

    # eval(): dropout을 끈다 / no_grad(): 평가에는 역전파가 필요 없으니 그래프를 만들지 않는다
    model.eval()
    with torch.no_grad():
        for batch in dataloader:
            input_images, labels = batch
            input_images, labels = input_images.to(device), labels.to(device)

            logits = model(input_images)
            predicted = torch.argmax(logits, dim=1)
            loss = criterion(logits, labels)

            avg_accuracy += (predicted == labels).sum()
            # .item() 은 배치마다 GPU->CPU 동기화가 걸리므로 텐서로 누적한다
            avg_loss += loss * input_images.size(0)

    avg_accuracy = (avg_accuracy / len(dataloader.dataset)).item()
    avg_loss = (avg_loss / len(dataloader.dataset)).item()

    return avg_loss, avg_accuracy

In [15]:
def train_one_epoch(model: nn.Module, dataloader: DataLoader, criterion: nn.Module,
                    optimizer: torch.optim.Optimizer, device: torch.device) -> Tuple[float, float]:
    avg_loss = torch.tensor(0.0, device=device)
    avg_accuracy = torch.tensor(0.0, device=device)

    model.train()
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        # detach 안 하면 epoch 내내 그래프가 안 풀려서 메모리가 쌓인다
        avg_loss += loss.detach() * images.size(0)
        avg_accuracy += (logits.detach().argmax(dim=1) == labels).sum()

    num_samples = len(dataloader.dataset)
    return (avg_loss / num_samples).item(), (avg_accuracy / num_samples).item()

In [18]:
for epoch in range(1, num_epoch + 1):
    current_lr = lr_scheduler.get_last_lr()[0]
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    lr_scheduler.step()

    print(f"[Epoch {epoch}/{num_epoch}] lr={current_lr:.4e} | "
          f"train loss {train_loss:.4f}, acc {train_acc*100:.2f}% | "
          f"test loss {test_loss:.4f}, acc {test_acc*100:.2f}%")

[Epoch 1/20] lr=2.5419e-04 | train loss 0.9008, acc 68.66% | test loss 0.7148, acc 75.31%
[Epoch 2/20] lr=2.2877e-04 | train loss 0.8886, acc 69.17% | test loss 0.6996, acc 75.74%
[Epoch 3/20] lr=2.0589e-04 | train loss 0.8807, acc 69.34% | test loss 0.6954, acc 75.57%
[Epoch 4/20] lr=1.8530e-04 | train loss 0.8660, acc 70.05% | test loss 0.6984, acc 75.41%
[Epoch 5/20] lr=1.6677e-04 | train loss 0.8562, acc 70.21% | test loss 0.6793, acc 76.12%
[Epoch 6/20] lr=1.5009e-04 | train loss 0.8524, acc 70.44% | test loss 0.6771, acc 76.35%
[Epoch 7/20] lr=1.3509e-04 | train loss 0.8388, acc 71.02% | test loss 0.6711, acc 76.48%
[Epoch 8/20] lr=1.2158e-04 | train loss 0.8346, acc 70.97% | test loss 0.6697, acc 76.65%
[Epoch 9/20] lr=1.0942e-04 | train loss 0.8307, acc 71.22% | test loss 0.6654, acc 76.71%
[Epoch 10/20] lr=9.8477e-05 | train loss 0.8267, acc 71.11% | test loss 0.6656, acc 76.79%
[Epoch 11/20] lr=8.8629e-05 | train loss 0.8233, acc 71.80% | test loss 0.6573, acc 77.12%
[Epoch 1

In [17]:
sum(p.numel() for p in model.parameters())

260160

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = model(images.to(device)).argmax(dim=1).cpu()

print("정답:", [label_names[i] for i in labels[:10].tolist()])
print("예측:", [label_names[i] for i in preds[:10].tolist()])